# 02 · Modalidades, cada una con su modelo

Cada archivo entra por un adaptador distinto. Si el modelo no está instalado, la celda lo dice y el resto del cuaderno sigue.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from helios.config import Settings
from helios.demo.case import demo_dir, load_demo_case
from helios.demo.content import fact_lines, risk_line
from helios.ingest.fields import extract_document
from helios.orchestration.router import route_file

settings = Settings.from_env()
case = load_demo_case()
for path in [case.pdf_path, case.chart_path, case.audio_path, case.news_path, case.prices_path]:
    print(route_file(path) if path else "ausente", path)


## Documento

La capa de texto manda en las cifras. La visión solo corre después, para describir; no para inventar un margen.

In [ ]:
extracted = extract_document("\n".join(fact_lines() + [risk_line()]))
print(extracted["fields"])
print(extracted["guidance_text"])
print(extracted["risk_text"])

if case.pdf_path:
    from helios.ingest.pdf import read_pdf
    document = read_pdf(case.pdf_path)
    print("páginas", len(document.pages), "campos", document.fields)


## Visión, transcripción, embeddings y CLIP

Estas celdas llaman al adaptador real. Si Ollama no responde o falta la librería, verás el motivo y puedes seguir al cuaderno de orquestación: la mesa tiene un modo degradado.

In [ ]:
from helios.models.ollama_client import OllamaClient

client = OllamaClient(settings)
print("ollama", client.healthy(), client.model_names() if client.healthy() else "apagado")
if client.healthy() and case.chart_path:
    model = client.resolve(settings.vision_model, settings.vision_fallbacks)
    print("visión", model)
    if model:
        text = client.chat(
            model,
            "Describe la tendencia, sin estimar precios exactos.",
            images=[case.chart_path.read_bytes()],
            timeout=180,
        )
        print(text[:500])


In [ ]:
if case.audio_path:
    try:
        from helios.models.whisper_asr import transcribe
        transcript, cached = transcribe(case.audio_path, settings)
        print(transcript.language, "caché" if cached else "frío")
        print(transcript.text[:400])
    except Exception as exc:
        print("transcripción no disponible:", exc)
else:
    print("sin audio en", demo_dir())
